# SGT runner — Kaggle (T4×2 / P100)

Use this for the 1.5B sweep and overflow runs. Internet must be ON in notebook settings.
Outputs go to `/kaggle/working/runs`; download the JSONs at session end.

In [ ]:
%cd /kaggle/working
!rm -rf prism
!git clone --depth 1 https://github.com/humanaiconvention/prism
%cd /kaggle/working/prism
!pip install -q -e . transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 bitsandbytes==0.44.1

In [ ]:
import os, sys, subprocess
sys.path.insert(0, '/kaggle/working/prism/src')
RUNS_DIR = '/kaggle/working/runs/1p5b'
os.makedirs(RUNS_DIR, exist_ok=True)
MODEL = 'Qwen/Qwen2.5-1.5B'
TEACHER = 'Qwen/Qwen2.5-7B'   # for R4 only; 4-bit auto via bitsandbytes

In [ ]:
# Dispatch one model-scale slice per session — Kaggle's 12-hour cap fits ~6 runs
BATCH = [
    # ('regime', seed)
    ('R1', 11), ('R1', 23), ('R1', 42),
    ('R2', 11), ('R2', 23), ('R2', 42),
]
for regime, seed in BATCH:
    cmd = ['python', 'sgt_runner.py', '--regime', regime, '--seed', str(seed),
           '--model', MODEL, '--out', RUNS_DIR]
    if regime == 'R4': cmd += ['--teacher', TEACHER]
    print('===', ' '.join(cmd)); subprocess.run(cmd, cwd='/kaggle/working/prism/experiments/sgt', check=True)

In [ ]:
# Verify outputs and zip for download
!ls -la {RUNS_DIR}
!cd /kaggle/working && zip -r runs_1p5b.zip runs/